In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Load dataset
data_path = Path("../data/canteen_sales.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (289525, 17)


,date,time_slot,day_of_week,month,item,category,price,quantity_sold,revenue,is_saturday,is_sunday,is_exam_period,is_holiday,weather,temperature,rainfall,college_event
0,2025-07-01,08:00:00,Tuesday,7,Vada Pav,Snacks,20,35,700,0,0,0,0,Cloudy,23.66,1.56,0
1,2025-07-01,08:00:00,Tuesday,7,Samosa,Snacks,15,33,495,0,0,0,0,Cloudy,23.66,1.56,0
2,2025-07-01,08:00:00,Tuesday,7,Kachori,Snacks,20,20,400,0,0,0,0,Cloudy,23.66,1.56,0
3,2025-07-01,08:00:00,Tuesday,7,Misal Pav,Snacks,50,15,750,0,0,0,0,Cloudy,23.66,1.56,0
4,2025-07-01,08:00:00,Tuesday,7,Bread Pakoda,Snacks,25,15,375,0,0,0,0,Cloudy,23.66,1.56,0


In [2]:
# =========================
# DATA PREPARATION
# =========================

# Convert date
df["date"] = pd.to_datetime(df["date"])

# Convert time
df["time_slot"] = pd.to_datetime(
    df["time_slot"],
    format="%H:%M:%S"
).dt.time

# Extract hour and minute
df["hour"] = df["time_slot"].apply(lambda x: x.hour)
df["minute"] = df["time_slot"].apply(lambda x: x.minute)

# Convert time into a numerical index
df["time_index"] = (
    df["hour"] * 60 +
    df["minute"]
)

print(df[[
    "time_slot",
    "hour",
    "minute",
    "time_index"
]].head())

  time_slot  hour  minute  time_index
0  08:00:00     8       0         480
1  08:00:00     8       0         480
2  08:00:00     8       0         480
3  08:00:00     8       0         480
4  08:00:00     8       0         480


In [3]:
# =========================
# FINAL FEATURES
# =========================

features = [
    "day_of_week",
    "month",
    "item",
    "category",
    "price",
    "is_saturday",
    "is_sunday",
    "is_exam_period",
    "is_holiday",
    "weather",
    "college_event",
    "hour",
    "minute",
    "time_index"
]

target = "quantity_sold"

X = df[features]
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nRevenue included:", "revenue" in X.columns)

Feature shape: (289525, 14)
Target shape: (289525,)

Features:
['day_of_week', 'month', 'item', 'category', 'price', 'is_saturday', 'is_sunday', 'is_exam_period', 'is_holiday', 'weather', 'college_event', 'hour', 'minute', 'time_index']

Revenue included: False


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (231620, 14)
Testing data: (57905, 14)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "day_of_week",
    "item",
    "category",
    "weather"
]

numeric_features = [
    "month",
    "price",
    "is_saturday",
    "is_sunday",
    "is_exam_period",
    "is_holiday",
    "college_event",
    "hour",
    "minute",
    "time_index"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", final_model)
    ]
)

In [7]:
print("Training final Random Forest model...")

model_pipeline.fit(
    X_train,
    y_train
)

print("Training completed successfully.")

Training final Random Forest model...
Training completed successfully.


In [8]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("FINAL MODEL PERFORMANCE")
print("-----------------------")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

FINAL MODEL PERFORMANCE
-----------------------
MAE  : 2.5138
RMSE : 4.1146
R²   : 0.9675


In [9]:
import joblib

model_path = Path("../models/final_demand_model.pkl")

joblib.dump(
    model_pipeline,
    model_path
)

print("Final model saved successfully!")
print(model_path)

Final model saved successfully!
..\models\final_demand_model.pkl
